# Модель SIR в сетях Петри

Данный скрипт реализует базовый эксперимент для модели SIR
с использованием сетей Петри. Выполняются детерминированная
и стохастическая симуляции, строятся графики динамики эпидемии.

## Подключение модулей

In [ ]:
using DrWatson
@quickactivate "project"
using Random
include(srcdir("SIRPetri.jl"))
using .SIRPetri
using DataFrames, CSV, Plots

## Параметры модели

- `β` (бета) — коэффициент заражения (скорость передачи инфекции)
- `γ` (гамма) — коэффициент выздоровления
- `tmax` — максимальное время симуляции

In [ ]:
β = 0.3
γ = 0.1
tmax = 100.0

## Создание сети Петри

Модель SIR содержит два перехода:
- `infection`: S + I → I + I (заражение)
- `recovery`: I → R (выздоровление)

In [ ]:
net, u0, states = build_sir_network(β, γ)

## Детерминированная симуляция (ODE)

Решение системы обыкновенных дифференциальных уравнений
методом Tsit5. Сохраняется в CSV и визуализируется.

In [ ]:
df_det = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.5, rates = [β, γ])
CSV.write(datadir("sir_det.csv"), df_det)

Детерминированный график

In [ ]:
p_det = plot_sir(df_det)
savefig(plotsdir("sir_det_dynamics.png"))

Отображение графика в Jupyter (если выполняется в ноутбуке)

In [ ]:
display(p_det)

## Стохастическая симуляция (алгоритм Гиллеспи)

Прямой метод Гиллеспи (SSA) для дискретных событий.
Фиксируем seed для воспроизводимости результатов.

In [ ]:
Random.seed!(123)
df_stoch = simulate_stochastic(net, u0, (0.0, tmax), rates = [β, γ])
CSV.write(datadir("sir_stoch.csv"), df_stoch)

Стохастический график

In [ ]:
p_stoch = plot_sir(df_stoch)
savefig(plotsdir("sir_stoch_dynamics.png"))

Отображение графика в Jupyter

In [ ]:
display(p_stoch)

## Вывод

- Детерминированная модель даёт гладкую усреднённую кривую
- Стохастическая модель показывает случайные флуктуации
- При большом количестве восприимчивых (990) траектории близки

In [ ]:
println("Базовый прогон завершён. Результаты в data/ и plots/")